In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EcommerceAnalysis") \
    .getOrCreate()

data_path = "hdfs://localhost:9000/bigdata/nhom11/data"

sales       = spark.read.csv(f"{data_path}/sales.csv",       header=True, inferSchema=True)
customers   = spark.read.csv(f"{data_path}/customers.csv",   header=True, inferSchema=True)
products    = spark.read.csv(f"{data_path}/products.csv",    header=True, inferSchema=True)
reviews     = spark.read.csv(f"{data_path}/reviews.csv",     header=True, inferSchema=True)
web_traffic = spark.read.csv(f"{data_path}/web_traffic.csv", header=True, inferSchema=True)
promotions  = spark.read.csv(f"{data_path}/promotions.csv",  header=True, inferSchema=True)
geography   = spark.read.csv(f"{data_path}/geography.csv",   header=True, inferSchema=True)
inventory  = spark.read.csv(f"{data_path}/inventory.csv",   header=True, inferSchema=True)
orders     = spark.read.csv(f"{data_path}/orders.csv",      header=True, inferSchema=True)
order_items= spark.read.csv(f"{data_path}/order_items.csv", header=True, inferSchema=True)
shipments  = spark.read.csv(f"{data_path}/shipments.csv",   header=True, inferSchema=True)
returns    = spark.read.csv(f"{data_path}/returns.csv",     header=True, inferSchema=True)
payments   = spark.read.csv(f"{data_path}/payments.csv",    header=True, inferSchema=True)

for name, df in [
    ("sales",        sales),
    ("customers",    customers),
    ("products",     products),
    ("reviews",      reviews),
    ("web_traffic",  web_traffic),
    ("promotions",   promotions),
    ("inventory",    inventory),
    ("orders",       orders),
    ("order_items",  order_items),
    ("shipments",    shipments),
    ("returns",      returns),
    ("payments",     payments),
    ("geography",    geography)
]:
    df.createOrReplaceTempView(name)



Q1. Doanh thu và lợi nhuận theo tháng của công ty

In [12]:
result = spark.sql("""
SELECT DATE_FORMAT(Date, 'yyyy-MM') AS month,
       ROUND(SUM(Revenue), 2) AS total_revenue,
       ROUND(SUM(COGS), 2) AS total_cogs,
       ROUND(SUM(Revenue - COGS), 2) AS gross_profit
FROM sales
GROUP BY month
ORDER BY month;
""")
result.show()
result.toPandas()


+-------+--------------+--------------+--------------+
|  month| total_revenue|    total_cogs|  gross_profit|
+-------+--------------+--------------+--------------+
|2012-07|1.3040677351E8|1.0251647387E8| 2.789029964E7|
|2012-08|1.5908924031E8|1.2558530643E8| 3.350393388E7|
|2012-09|1.2930713382E8|1.0259615482E8|   2.6710979E7|
|2012-10|1.1018569445E8| 8.805480503E7| 2.213088942E7|
|2012-11| 9.818629524E7| 7.782007741E7| 2.036621783E7|
|2012-12|1.1432261069E8| 9.088910625E7| 2.343350444E7|
|2013-01| 9.130867703E7| 7.302380408E7| 1.828487295E7|
|2013-02|1.0978313794E8| 9.028687659E7| 1.949626135E7|
|2013-03|1.5183465727E8|1.2989778243E8| 2.193687484E7|
|2013-04|1.9892693209E8| 1.655819613E8| 3.334497079E7|
|2013-05|2.0050010948E8| 1.597591538E8| 4.074095568E7|
|2013-06|1.9851762854E8|1.6630256172E8| 3.221506682E7|
|2013-07|1.6574535399E8|1.5304461389E8|  1.27007401E7|
|2013-08|1.1627175596E8|  1.51947691E8|-3.567593504E7|
|2013-09|1.2753414797E8|1.1616133982E8| 1.137280815E7|
|2013-10|1

,month,total_revenue,total_cogs,gross_profit
0,2012-07,1.304068e+08,1.025165e+08,27890299.64
1,2012-08,1.590892e+08,1.255853e+08,33503933.88
2,2012-09,1.293071e+08,1.025962e+08,26710979.00
3,2012-10,1.101857e+08,8.805481e+07,22130889.42
4,2012-11,9.818630e+07,7.782008e+07,20366217.83
...,...,...,...,...
121,2022-08,1.135429e+08,9.403338e+07,19509565.13
122,2022-09,8.576429e+07,7.839524e+07,7369044.32
123,2022-10,7.519445e+07,6.196989e+07,13224558.51
124,2022-11,5.220008e+07,4.709802e+07,5102060.14


Vào tháng 2013-08, doanh thu đạt 116.27 triệu nhưng chi phí giá vốn (total_cogs) vọt lên tới 151.94 triệu, dẫn đến lợi nhuận gộp bị âm nặng (-35.67 triệu).
Ý nghĩa kinh doanh: Đây là hiện tượng kinh doanh "bán lỗ".
Nguyên nhân giả định: Doanh nghiệp đã chạy một chiến dịch xả hàng tồn kho quá tay, hoặc áp dụng các mã giảm giá (Promotions) quá sâu, cắt máu trực tiếp vào giá vốn sản phẩm.
Hành động thúc đẩy: Bộ phận quản trị chiến lược cần thắt chặt lại quy định phân bổ mã giảm giá, không cho phép áp dụng voucher cộng dồn khiến giá bán thấp hơn giá nhập hàng (COGS).
Số liệu: Trong quý 2 năm 2013, doanh thu đạt mức kỷ lục, duy trì đều đặn ở ngưỡng gần 200 triệu mỗi tháng. Đặc biệt, tháng 2013-05 đạt đỉnh lợi nhuận gộp lên tới 40.74 triệu (Biên lợi nhuận gộp ~20.3%).
Ý nghĩa kinh doanh: Đây là giai đoạn thị trường bùng nổ hoặc doanh nghiệp có các chiến dịch marketing/sản phẩm mới cực kỳ thành công. Giá vốn được tối ưu tốt, khách hàng chấp nhận mua sản phẩm với giá trị cao. Doanh nghiệp nên lưu lại lịch sử cấu trúc danh mục sản phẩm (Product Mix) của giai đoạn này để áp dụng tái bản cho các năm sau.
Hiện tượng "Doanh thu bẫy" vào Tháng 12/2013
Số liệu: Tháng 2013-12 là mùa mua sắm cuối năm, doanh thu hồi phục nhẹ đạt 91.28 triệu, nhưng lợi nhuận gộp (gross_profit) gần như bằng 0, chỉ vỏn vẹn 1.11 triệu.
Ý nghĩa kinh doanh: Giống như tháng 8/2013, doanh nghiệp lại rơi vào cái bẫy "tăng trưởng ảo". Doanh số có tăng so với tháng trước (90.37 triệu), nhưng chi phí bỏ ra quá lớn (90.17 triệu). Việc khuyến mãi cuối năm để lấy volume (sản lượng) đã xóa sổ hoàn toàn lợi nhuận của doanh nghiệp.

In [14]:
result_q1 = spark.sql("""
SELECT
    DATE_FORMAT(Date, 'yyyy-MM') AS month,
    CAST(SUM(Revenue) AS DECIMAL(18, 2)) AS total_revenue,
    CAST(SUM(COGS) AS DECIMAL(18, 2)) AS total_cogs,
    CAST(SUM(Revenue - COGS) AS DECIMAL(18, 2)) AS gross_profit
FROM
    sales
GROUP BY 1
ORDER BY month;
""")
result_q1.show(truncate=False)

import pandas as pd

pdf = result_q1.toPandas()

pdf['total_revenue'] = pdf['total_revenue'].apply(lambda x: "{:,.2f}".format(float(x)))
pdf['total_cogs'] = pdf['total_cogs'].apply(lambda x: "{:,.2f}".format(float(x)))
pdf['gross_profit'] = pdf['gross_profit'].apply(lambda x: "{:,.2f}".format(float(x)))

pdf

+-------+-------------+------------+------------+
|month  |total_revenue|total_cogs  |gross_profit|
+-------+-------------+------------+------------+
|2012-07|130406773.51 |102516473.87|27890299.64 |
|2012-08|159089240.31 |125585306.43|33503933.88 |
|2012-09|129307133.82 |102596154.82|26710979.00 |
|2012-10|110185694.45 |88054805.03 |22130889.42 |
|2012-11|98186295.24  |77820077.41 |20366217.83 |
|2012-12|114322610.69 |90889106.25 |23433504.44 |
|2013-01|91308677.03  |73023804.08 |18284872.95 |
|2013-02|109783137.94 |90286876.59 |19496261.35 |
|2013-03|151834657.27 |129897782.43|21936874.84 |
|2013-04|198926932.09 |165581961.30|33344970.79 |
|2013-05|200500109.48 |159759153.80|40740955.68 |
|2013-06|198517628.54 |166302561.72|32215066.82 |
|2013-07|165745353.99 |153044613.89|12700740.10 |
|2013-08|116271755.96 |151947691.00|-35675935.04|
|2013-09|127534147.97 |116161339.82|11372808.15 |
|2013-10|115089836.74 |92685814.37 |22404022.37 |
|2013-11|90371311.82  |77118230.78 |13253081.04 |


,month,total_revenue,total_cogs,gross_profit
0,2012-07,"130,406,773.51","102,516,473.87","27,890,299.64"
1,2012-08,"159,089,240.31","125,585,306.43","33,503,933.88"
2,2012-09,"129,307,133.82","102,596,154.82","26,710,979.00"
3,2012-10,"110,185,694.45","88,054,805.03","22,130,889.42"
4,2012-11,"98,186,295.24","77,820,077.41","20,366,217.83"
...,...,...,...,...
121,2022-08,"113,542,943.47","94,033,378.34","19,509,565.13"
122,2022-09,"85,764,286.59","78,395,242.27","7,369,044.32"
123,2022-10,"75,194,452.31","61,969,893.80","13,224,558.51"
124,2022-11,"52,200,081.64","47,098,021.50","5,102,060.14"


Q2.Phân phối khách hàng theo kenh

In [15]:
result_q2 = spark.sql("""
SELECT acquisition_channel,
       COUNT(*) AS customer_count,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct
FROM customers
GROUP BY acquisition_channel
ORDER BY customer_count DESC;
                   """)
result_q2.show(truncate=False)
result_q2.toPandas()

26/06/13 19:13:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:13:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:13:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:13:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:13:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:13:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 1

+-------------------+--------------+-----+
|acquisition_channel|customer_count|pct  |
+-------------------+--------------+-----+
|organic_search     |36450         |29.89|
|social_media       |24448         |20.05|
|paid_search        |24285         |19.92|
|email_campaign     |14674         |12.03|
|referral           |12270         |10.06|
|direct             |9803          |8.04 |
+-------------------+--------------+-----+



26/06/13 19:13:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:13:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:13:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:13:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


,acquisition_channel,customer_count,pct
0,organic_search,36450,29.89
1,social_media,24448,20.05
2,paid_search,24285,19.92
3,email_campaign,14674,12.03
4,referral,12270,10.06
5,direct,9803,8.04


Kênh Organic Search (Tìm kiếm tự nhiên) dẫn đầu tuyệt đối: Chiếm lượng khách hàng lớn nhất (~33.2 tỷ). Điều này chứng tỏ thương hiệu đang làm rất tốt về SEO (tối ưu hóa công cụ tìm kiếm) hoặc độ nhận diện thương hiệu tự nhiên trên thị trường rất mạnh. Khách hàng chủ động tìm kiếm sản phẩm và chuyển đổi thành user của hệ thống.
Kênh Paid Search (Quảng cáo trả phí) bám đuổi sát sao: Đứng vị trí thứ hai (~25 tỷ). Doanh nghiệp đang đổ một lượng ngân sách lớn vào quảng cáo (như Google Ads). Kênh này mang lại hiệu quả chuyển đổi nhanh chóng nhưng đi kèm chi phí vận hành (CAC - Customer Acquisition Cost) cao.
Kênh Referral (Giới thiệu) có hiệu quả thấp nhất: Chỉ đạt ~8.3 tỷ khách hàng. Điều này phản ánh các chương trình tiếp thị liên kết (Affiliate), chính sách "Mời bạn bè nhận quà" (Referral Program) của hệ thống hiện tại chưa đủ hấp dẫn để kích thích người dùng chia sẻ. Đây là điểm yếu cần cải thiện trong quý tới.

Q3.Top sản phẩm biên lợi nhuận cao nhất

In [16]:
result_q3 = spark.sql("""
SELECT product_name, category, price,
       ROUND(price - cogs, 2) AS margin,
       ROUND((price - cogs) / price * 100, 2) AS margin_pct
FROM products
ORDER BY margin_pct DESC
LIMIT 20;
                   """)

result_q3.show(truncate=False)
result_q3.toPandas()

+-----------------+----------+------------------+-------+----------+
|product_name     |category  |price             |margin |margin_pct|
+-----------------+----------+------------------+-------+----------+
|NamStyle UC-01   |Streetwear|2708.37           |1353.1 |49.96     |
|VietMode RS-07   |Outdoor   |5039.37           |2516.66|49.94     |
|HanoiStreet UC-07|Streetwear|8971.2            |4479.32|49.93     |
|HanoiStreet UC-52|Streetwear|5262.4712658227845|2623.87|49.86     |
|HanoiStreet RP-23|Outdoor   |4615.655408483896 |2300.9 |49.85     |
|MekongFit RP-45  |Outdoor   |497.07            |247.79 |49.85     |
|VietMotion YY-11 |GenZ      |3733.948733944954 |1859.51|49.8      |
|PhoenixWear MA-01|Casual    |6299.37           |3132.05|49.72     |
|VietMode RP-05   |Outdoor   |4848.653793103448 |2409.3 |49.69     |
|HanoiStreet UE-42|Streetwear|6929.370000000001 |3441.13|49.66     |
|UrbanVN UE-20    |Streetwear|4438.989402985075 |2204.4 |49.66     |
|LotusWear UR-09  |Streetwear|6401

,product_name,category,price,margin,margin_pct
0,NamStyle UC-01,Streetwear,2708.370000,1353.10,49.96
1,VietMode RS-07,Outdoor,5039.370000,2516.66,49.94
2,HanoiStreet UC-07,Streetwear,8971.200000,4479.32,49.93
3,HanoiStreet UC-52,Streetwear,5262.471266,2623.87,49.86
4,HanoiStreet RP-23,Outdoor,4615.655408,2300.90,49.85
5,MekongFit RP-45,Outdoor,497.070000,247.79,49.85
6,VietMotion YY-11,GenZ,3733.948734,1859.51,49.80
7,PhoenixWear MA-01,Casual,6299.370000,3132.05,49.72
8,VietMode RP-05,Outdoor,4848.653793,2409.30,49.69
9,HanoiStreet UE-42,Streetwear,6929.370000,3441.13,49.66


Q4.Hiệu suất web traffic theo nguồn(web_traffic.csv)

In [17]:
result_q4 = spark.sql("""
SELECT traffic_source,
       SUM(sessions) AS total_sessions,
       ROUND(AVG(bounce_rate), 4) AS avg_bounce_rate,
       ROUND(AVG(avg_session_duration_sec), 1) AS avg_duration_sec
FROM web_traffic
GROUP BY traffic_source
ORDER BY total_sessions DESC;
                   """)
result_q4.show(truncate=False)
result_q4.toPandas()

+--------------+--------------+---------------+----------------+
|traffic_source|total_sessions|avg_bounce_rate|avg_duration_sec|
+--------------+--------------+---------------+----------------+
|organic_search|27196976      |0.0045         |211.2           |
|paid_search   |19598271      |0.0045         |209.4           |
|social_media  |15816226      |0.0045         |210.3           |
|email_campaign|12792670      |0.0045         |213.2           |
|referral      |9476845       |0.0045         |207.6           |
|direct        |6571549       |0.0045         |207.7           |
+--------------+--------------+---------------+----------------+



,traffic_source,total_sessions,avg_bounce_rate,avg_duration_sec
0,organic_search,27196976,0.0045,211.2
1,paid_search,19598271,0.0045,209.4
2,social_media,15816226,0.0045,210.3
3,email_campaign,12792670,0.0045,213.2
4,referral,9476845,0.0045,207.6
5,direct,6571549,0.0045,207.7


Q5.Phân bố khách hàng theo vùng địa lý (customers JOIN geography)

In [18]:
result_q5 = spark.sql("""
SELECT g.region, g.city,
       COUNT(c.customer_id) AS total_customers,
       COUNT(CASE WHEN c.gender = 'Female' THEN 1 END) AS female_count
FROM customers c
JOIN geography g ON c.zip = g.zip
GROUP BY g.region, g.city
ORDER BY total_customers DESC;
                   """)
result_q5.show(truncate=False)
result_q5.toPandas()

+-------+-------------------+---------------+------------+
|region |city               |total_customers|female_count|
+-------+-------------------+---------------+------------+
|East   |Cam Pha            |4398           |2170        |
|East   |Thai Nguyen        |4347           |2171        |
|East   |Phu Ly             |4243           |2138        |
|East   |Hanoi              |4240           |2100        |
|East   |Ha Long            |4236           |2031        |
|East   |Bac Ninh           |4172           |2017        |
|East   |Hai Phong          |4170           |2029        |
|East   |Nam Dinh           |4169           |2064        |
|East   |Bac Giang          |4160           |2014        |
|East   |Ninh Binh          |4081           |2011        |
|East   |Son Tay            |4075           |2017        |
|East   |Viet Tri           |4054           |1941        |
|East   |Uong Bi            |4026           |1959        |
|Central|Dong Hoi           |3912           |1877       

,region,city,total_customers,female_count
0,East,Cam Pha,4398,2170
1,East,Thai Nguyen,4347,2171
2,East,Phu Ly,4243,2138
3,East,Hanoi,4240,2100
4,East,Ha Long,4236,2031
5,East,Bac Ninh,4172,2017
6,East,Hai Phong,4170,2029
7,East,Nam Dinh,4169,2064
8,East,Bac Giang,4160,2014
9,East,Ninh Binh,4081,2011


Q6.Xếp hạng sản phẩm trong từng category (products JOIN reviews)

In [19]:
result_q6 = spark.sql("""
SELECT product_name, category,
       ROUND(AVG(rating), 2) AS avg_rating,
       COUNT(*) AS review_count,
       RANK() OVER (PARTITION BY category ORDER BY AVG(rating) DESC) AS rank_in_category,
       DENSE_RANK() OVER (ORDER BY AVG(rating) DESC) AS overall_rank
FROM reviews r
JOIN products p ON r.product_id = p.product_id
GROUP BY product_name, category;""")
result_q6.show(truncate=False)
result_q6.toPandas()

26/06/13 19:14:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:14:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:14:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:14:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:14:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:14:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 1

+-----------------+--------+----------+------------+----------------+------------+
|product_name     |category|avg_rating|review_count|rank_in_category|overall_rank|
+-----------------+--------+----------+------------+----------------+------------+
|DragonWear MA-40 |Casual  |5.0       |2           |1               |1           |
|DragonWear MA-10 |Casual  |5.0       |3           |1               |1           |
|VietMode MA-21   |Casual  |5.0       |1           |1               |1           |
|DragonWear MA-12 |Casual  |5.0       |1           |1               |1           |
|VietMode MA-24   |Casual  |5.0       |1           |1               |1           |
|VietMode MP-17   |Casual  |5.0       |1           |1               |1           |
|SaigonCore MA-04 |Casual  |5.0       |1           |1               |1           |
|DragonWear MA-23 |Casual  |4.75      |4           |8               |3           |
|VietMode MA-07   |Casual  |4.75      |4           |8               |3           |
|Mek

26/06/13 19:14:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:14:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:14:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:14:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


,product_name,category,avg_rating,review_count,rank_in_category,overall_rank
0,DragonWear MA-40,Casual,5.00,2,1,1
1,DragonWear MA-10,Casual,5.00,3,1,1
2,VietMode MA-21,Casual,5.00,1,1,1
3,DragonWear MA-12,Casual,5.00,1,1,1
4,VietMode MA-24,Casual,5.00,1,1,1
...,...,...,...,...,...,...
1274,NamStyle UE-01,Streetwear,2.00,1,725,603
1275,MekongFit UE-31,Streetwear,2.00,1,725,603
1276,VietMode UE-13,Streetwear,2.00,1,725,603
1277,SaigonCore UC-02,Streetwear,1.67,3,732,604


Q7. Phân khúc khách hàng theo 3 chỉ số quan trọng (Recency–Frequency–Monetary)

In [21]:
result_q7 = spark.sql("""
WITH frequency AS (
  SELECT customer_id, COUNT(order_id) AS order_count,
         MAX(review_date) AS last_order_date,
         DATEDIFF(CURRENT_DATE(), MAX(review_date)) AS recency_days
  FROM reviews
  GROUP BY customer_id
),
rfm AS (
  SELECT f.*, c.age_group, c.acquisition_channel,
         NTILE(4) OVER (ORDER BY recency_days ASC)    AS R_score,
         NTILE(4) OVER (ORDER BY order_count DESC)    AS F_score
  FROM frequency f
  JOIN customers c ON f.customer_id = c.customer_id
)
SELECT age_group, acquisition_channel,
       ROUND(AVG(R_score + F_score), 2) AS avg_rfm_score,
       COUNT(*) AS customer_count
FROM rfm
GROUP BY age_group, acquisition_channel
ORDER BY avg_rfm_score DESC
""")

result_q7.show(truncate=False)
result_q7.toPandas()

26/06/13 19:15:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:15:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:15:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:15:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:15:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:15:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 1

+---------+-------------------+-------------+--------------+
|age_group|acquisition_channel|avg_rfm_score|customer_count|
+---------+-------------------+-------------+--------------+
|18-24    |direct             |5.19         |545           |
|45-54    |referral           |5.07         |920           |
|35-44    |referral           |5.07         |1246          |
|35-44    |email_campaign     |5.07         |1567          |
|55+      |organic_search     |5.05         |1637          |
|55+      |social_media       |5.05         |1108          |
|45-54    |direct             |5.04         |757           |
|25-34    |social_media       |5.03         |2848          |
|55+      |paid_search        |5.03         |1088          |
|55+      |email_campaign     |5.03         |639           |
|55+      |direct             |5.03         |447           |
|25-34    |organic_search     |5.02         |4319          |
|45-54    |organic_search     |5.02         |2807          |
|55+      |referral     

26/06/13 19:15:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:15:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:15:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:15:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:15:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:15:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 1

,age_group,acquisition_channel,avg_rfm_score,customer_count
0,18-24,direct,5.19,545
1,45-54,referral,5.07,920
2,35-44,referral,5.07,1246
3,35-44,email_campaign,5.07,1567
4,55+,organic_search,5.05,1637
5,55+,social_media,5.05,1108
6,45-54,direct,5.04,757
7,25-34,social_media,5.03,2848
8,55+,paid_search,5.03,1088
9,55+,email_campaign,5.03,639


Q8.Đánh giá sản phẩm theo vùng

In [22]:
result_q8 = spark.sql("""
WITH traffic_monthly AS (
  SELECT DATE_FORMAT(date, 'yyyy-MM') AS month,
         traffic_source, SUM(sessions) AS sessions
  FROM web_traffic GROUP BY month, traffic_source
),
region_revenue AS (
  SELECT DATE_FORMAT(s.Date, 'yyyy-MM') AS month,
         g.region, SUM(s.Revenue) AS revenue, SUM(s.COGS) AS cogs
  FROM sales s
  JOIN customers c ON DATE_FORMAT(s.Date,'yyyy-MM') = DATE_FORMAT(c.signup_date,'yyyy-MM')
  JOIN geography g ON c.zip = g.zip
  GROUP BY month, g.region
)
SELECT t.month, t.traffic_source, t.sessions,
       r.region, ROUND(r.revenue, 2) AS revenue,
       ROUND(r.revenue / NULLIF(t.sessions, 0), 2) AS revenue_per_session
FROM traffic_monthly t
JOIN region_revenue r ON t.month = r.month
ORDER BY t.month, revenue_per_session DESC;
                   """)

result_q8.show(truncate=False)
result_q8.toPandas()

+-------+--------------+--------+-------+---------------+-------------------+
|month  |traffic_source|sessions|region |revenue        |revenue_per_session|
+-------+--------------+--------+-------+---------------+-------------------+
|2013-01|referral      |28134   |East   |8.30908960973E9|295339.79          |
|2013-01|social_media  |28776   |East   |8.30908960973E9|288750.68          |
|2013-01|direct        |37852   |East   |8.30908960973E9|219515.21          |
|2013-01|referral      |28134   |Central|5.20459459071E9|184993.05          |
|2013-01|social_media  |28776   |Central|5.20459459071E9|180865.81          |
|2013-01|email_campaign|49649   |East   |8.30908960973E9|167356.64          |
|2013-01|direct        |37852   |Central|5.20459459071E9|137498.54          |
|2013-01|paid_search   |72457   |East   |8.30908960973E9|114676.15          |
|2013-01|organic_search|78138   |East   |8.30908960973E9|106338.65          |
|2013-01|email_campaign|49649   |Central|5.20459459071E9|104827.

,month,traffic_source,sessions,region,revenue,revenue_per_session
0,2013-01,referral,28134,East,8.309090e+09,295339.79
1,2013-01,social_media,28776,East,8.309090e+09,288750.68
2,2013-01,direct,37852,East,8.309090e+09,219515.21
3,2013-01,referral,28134,Central,5.204595e+09,184993.05
4,2013-01,social_media,28776,Central,5.204595e+09,180865.81
...,...,...,...,...,...,...
2083,2022-12,paid_search,102170,West,1.641840e+10,160696.83
2084,2022-12,organic_search,221159,Central,3.462026e+10,156540.13
2085,2022-12,email_campaign,106602,West,1.641840e+10,154015.83
2086,2022-12,social_media,127109,West,1.641840e+10,129167.84


Q9.Phân tích tồn kho toàn diện theo category & segment

In [23]:
result_q9 = spark.sql("""
WITH monthly_inventory AS (
    SELECT
        year, month, category, segment,
        SUM(stock_on_hand)                                  AS total_stock,
        SUM(units_received)                                 AS total_received,
        SUM(units_sold)                                     AS total_sold,
        SUM(stockout_days)                                  AS total_stockout_days,
        ROUND(AVG(days_of_supply), 1)                       AS avg_days_of_supply,
        ROUND(AVG(fill_rate) * 100, 2)                      AS avg_fill_rate_pct,
        ROUND(AVG(sell_through_rate) * 100, 2)              AS avg_sell_through_pct,
        SUM(stockout_flag)                                  AS stockout_products,
        SUM(overstock_flag)                                 AS overstock_products,
        SUM(reorder_flag)                                   AS need_reorder_products,
        COUNT(DISTINCT product_id)                          AS total_products
    FROM inventory
    GROUP BY year, month, category, segment
),
with_status_ratio AS (
    SELECT *,
        ROUND(stockout_products  * 100.0 / total_products, 2) AS stockout_rate_pct,
        ROUND(overstock_products * 100.0 / total_products, 2) AS overstock_rate_pct,
        ROUND(total_sold * 100.0 / NULLIF(total_received + total_stock, 0), 2) AS inventory_turnover_pct,
        RANK() OVER (
            PARTITION BY year, month
            ORDER BY avg_fill_rate_pct DESC
        )                                                   AS fill_rate_rank
    FROM monthly_inventory
)
SELECT
    year, month, category, segment,
    total_stock,
    total_sold,
    avg_days_of_supply,
    avg_fill_rate_pct,
    avg_sell_through_pct,
    stockout_rate_pct,
    overstock_rate_pct,
    inventory_turnover_pct,
    need_reorder_products,
    fill_rate_rank,
    CASE
        WHEN avg_sell_through_pct >= 80 AND stockout_rate_pct < 10  THEN 'Healthy'
        WHEN stockout_rate_pct >= 30                                 THEN 'Critical Stockout'
        WHEN overstock_rate_pct >= 40                                THEN 'Overstock Risk'
        WHEN need_reorder_products > total_products * 0.5           THEN 'Reorder Urgent'
        ELSE                                                              'Monitor'
    END                                                         AS inventory_health
FROM with_status_ratio
ORDER BY year, month, stockout_rate_pct DESC
""")

result_q9.show(30, truncate=False)
result_q9.toPandas()

+----+-----+----------+-----------+-----------+----------+------------------+-----------------+--------------------+-----------------+------------------+----------------------+---------------------+--------------+-----------------+
|year|month|category  |segment    |total_stock|total_sold|avg_days_of_supply|avg_fill_rate_pct|avg_sell_through_pct|stockout_rate_pct|overstock_rate_pct|inventory_turnover_pct|need_reorder_products|fill_rate_rank|inventory_health |
+----+-----+----------+-----------+-----------+----------+------------------+-----------------+--------------------+-----------------+------------------+----------------------+---------------------+--------------+-----------------+
|2012|7    |GenZ      |Trendy     |1314       |416       |94.7              |95.78            |24.06               |80.00            |86.67             |23.16                 |0                    |8             |Critical Stockout|
|2012|7    |Streetwear|Performance|2928       |922       |93.0          

,year,month,category,segment,total_stock,total_sold,avg_days_of_supply,avg_fill_rate_pct,avg_sell_through_pct,stockout_rate_pct,overstock_rate_pct,inventory_turnover_pct,need_reorder_products,fill_rate_rank,inventory_health
0,2012,7,GenZ,Trendy,1314,416,94.7,95.78,24.06,80.00,86.67,23.16,0,8,Critical Stockout
1,2012,7,Streetwear,Performance,2928,922,93.0,96.32,24.42,75.44,54.39,22.98,0,6,Critical Stockout
2,2012,7,Casual,All-weather,497,158,92.9,96.04,24.43,75.00,56.25,23.30,0,7,Critical Stockout
3,2012,7,Streetwear,Everyday,7675,2401,94.5,96.43,24.11,71.76,78.82,22.76,0,5,Critical Stockout
4,2012,7,Streetwear,Balanced,2589,816,92.4,96.52,24.53,64.44,44.44,23.01,0,4,Critical Stockout
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1092,2022,12,Streetwear,Everyday,19466,406,3319.0,96.25,7.79,70.83,94.44,2.04,0,5,Critical Stockout
1093,2022,12,Streetwear,Balanced,9105,623,713.2,96.67,14.43,62.71,76.27,6.34,0,4,Critical Stockout
1094,2022,12,GenZ,Trendy,5126,199,1019.4,97.20,11.75,56.82,84.09,3.72,0,3,Critical Stockout
1095,2022,12,Streetwear,Standard,553,58,437.3,97.56,12.49,53.33,80.00,9.39,0,1,Critical Stockout


Nhìn vào inventory_health để ưu tiên xử lý — Critical Stockout mất doanh thu ngay lập tức, Overstock Risk gây đọng vốn, Reorder Urgent cần đặt hàng nhà cung cấp sớm.

Q10.Cohort Retention: Tỷ lệ khách hàng quay lại theo năm đăng ký. Cohort năm nào có retention_rate_pct giảm mạnh sau Year 1 cần xem lại chiến lược giữ chân khách hàng. Phát hiện những ngày doanh thu bất thường (tăng đột biến hoặc giảm mạnh hơn 2 độ lệch chuẩn) — thường trùng với sự kiện khuyến mãi hoặc sự cố hệ thống.

In [24]:
result_q10 = spark.sql("""
    WITH customer_cohort AS (
        SELECT
            c.customer_id,
            YEAR(c.signup_date)                        AS cohort_year,
            YEAR(r.review_date)                        AS activity_year,
            COUNT(r.review_id)                         AS orders_in_year
        FROM customers c
        JOIN reviews r ON c.customer_id = r.customer_id
        GROUP BY c.customer_id, YEAR(c.signup_date), YEAR(r.review_date)
    ),
    cohort_size AS (
        SELECT cohort_year, COUNT(DISTINCT customer_id) AS total_customers
        FROM customer_cohort
        GROUP BY cohort_year
    ),
    cohort_activity AS (
        SELECT
            cohort_year,
            activity_year,
            (activity_year - cohort_year)              AS year_since_signup,
            COUNT(DISTINCT customer_id)                AS active_customers
        FROM customer_cohort
        GROUP BY cohort_year, activity_year
    )
    SELECT
        ca.cohort_year,
        ca.year_since_signup,
        ca.active_customers,
        cs.total_customers,
        ROUND(ca.active_customers * 100.0 / cs.total_customers, 2) AS retention_rate_pct
    FROM cohort_activity ca
    JOIN cohort_size cs ON ca.cohort_year = cs.cohort_year
    WHERE ca.year_since_signup >= 0
    ORDER BY ca.cohort_year, ca.year_since_signup
""")

result_q10.show(50)
result_q10.toPandas()

+-----------+-----------------+----------------+---------------+------------------+
|cohort_year|year_since_signup|active_customers|total_customers|retention_rate_pct|
+-----------+-----------------+----------------+---------------+------------------+
|       2012|                0|              29|            379|              7.65|
|       2012|                1|              78|            379|             20.58|
|       2012|                2|              99|            379|             26.12|
|       2012|                3|             112|            379|             29.55|
|       2012|                4|              92|            379|             24.27|
|       2012|                5|              85|            379|             22.43|
|       2012|                6|              81|            379|             21.37|
|       2012|                7|              45|            379|             11.87|
|       2012|                8|              39|            379|            

,cohort_year,year_since_signup,active_customers,total_customers,retention_rate_pct
0,2012,0,29,379,7.65
1,2012,1,78,379,20.58
2,2012,2,99,379,26.12
3,2012,3,112,379,29.55
4,2012,4,92,379,24.27
...,...,...,...,...,...
61,2020,1,785,6910,11.36
62,2020,2,761,6910,11.01
63,2021,0,792,7568,10.47
64,2021,1,847,7568,11.19


Q11.Doanh thu trung bình động 30 ngày + phát hiện bất thường

In [25]:
result_q11 = spark.sql("""
    WITH daily_revenue AS (
        SELECT
            Date,
            Revenue,
            COGS,
            ROUND(Revenue - COGS, 2)                   AS daily_profit
        FROM sales
    ),
    moving_avg AS (
        SELECT
            Date,
            Revenue,
            daily_profit,
            ROUND(AVG(Revenue) OVER (
                ORDER BY Date
                ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
            ), 2)                                      AS ma30_revenue,
            ROUND(STDDEV(Revenue) OVER (
                ORDER BY Date
                ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
            ), 2)                                      AS stddev30_revenue,
            ROUND(AVG(daily_profit) OVER (
                ORDER BY Date
                ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
            ), 2)                                      AS ma7_profit
        FROM daily_revenue
    )
    SELECT
        Date,
        ROUND(Revenue, 2)                              AS revenue,
        ma30_revenue,
        ma7_profit,
        ROUND(Revenue - ma30_revenue, 2)               AS deviation_from_ma30,
        CASE
            WHEN Revenue > ma30_revenue + 2 * stddev30_revenue THEN 'Spike'
            WHEN Revenue < ma30_revenue - 2 * stddev30_revenue THEN 'Drop'
            ELSE 'Normal'
        END                                            AS anomaly_flag
    FROM moving_avg
    WHERE stddev30_revenue IS NOT NULL
    ORDER BY Date
""")

result_q11.filter("anomaly_flag != 'Normal'").show(20)

+----------+-------------+------------+----------+-------------------+------------+
|      Date|      revenue|ma30_revenue|ma7_profit|deviation_from_ma30|anomaly_flag|
+----------+-------------+------------+----------+-------------------+------------+
|2012-07-31|   9151622.14|  4657384.77|1264908.52|         4494237.37|       Spike|
|2012-08-01|    9148357.3|  4812245.89| 1369647.7|         4336111.41|       Spike|
|2012-08-02|    9692427.0|  4974918.59|1495666.63|         4717508.41|       Spike|
|2012-08-03|   9297269.95|  5114042.66|1640118.66|         4183227.29|       Spike|
|2012-08-28|   9508676.62|  4879174.08|1096418.81|         4629502.54|       Spike|
|2012-08-30| 1.00865341E7|  4999485.39|1345210.26|         5087048.71|       Spike|
|2012-10-28|   6259883.83|  3448998.56| 734325.51|         2810885.27|       Spike|
|2012-10-29|    5424539.1|  3453004.05| 792504.97|         1971535.05|       Spike|
|2012-10-30|   6347274.89|  3508100.55| 856805.48|         2839174.34|      

26/06/13 19:17:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:17:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:17:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:17:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 19:17:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Q12.Multi-dimensional Funnel: Traffic → Engagement → Purchase theo kênh & vùng. Kênh nào có visitor_to_buyer_pct thấp nhưng avg_session_duration_sec cao → khách xem nhiều nhưng không mua → cần cải thiện UX hoặc pricing.

In [26]:
result_q12 = spark.sql("""
    WITH monthly_traffic AS (
        SELECT
            DATE_FORMAT(date, 'yyyy-MM')               AS month,
            traffic_source,
            SUM(sessions)                              AS sessions,
            SUM(unique_visitors)                       AS unique_visitors,
            SUM(page_views)                            AS page_views,
            ROUND(AVG(bounce_rate) * 100, 2)           AS avg_bounce_pct,
            ROUND(AVG(avg_session_duration_sec), 1)    AS avg_duration_sec
        FROM web_traffic
        GROUP BY month, traffic_source
    ),
    monthly_orders AS (
        SELECT
            DATE_FORMAT(r.review_date, 'yyyy-MM')      AS month,
            g.region,
            COUNT(DISTINCT r.order_id)                 AS total_orders,
            COUNT(DISTINCT r.customer_id)              AS unique_buyers,
            ROUND(AVG(r.rating), 2)                    AS avg_rating
        FROM reviews r
        JOIN customers c  ON r.customer_id = c.customer_id
        JOIN geography g  ON c.zip = g.zip
        GROUP BY month, g.region
    ),
    monthly_revenue AS (
        SELECT
            DATE_FORMAT(Date, 'yyyy-MM')               AS month,
            ROUND(SUM(Revenue), 2)                     AS revenue,
            ROUND(SUM(COGS), 2)                        AS cogs
        FROM sales
        GROUP BY month
    )
    SELECT
        t.month,
        t.traffic_source,
        o.region,
        t.sessions,
        t.unique_visitors,
        o.unique_buyers,
        o.total_orders,
        r.revenue,
        ROUND(o.unique_buyers * 100.0 / NULLIF(t.unique_visitors, 0), 2) AS visitor_to_buyer_pct,
        ROUND(r.revenue / NULLIF(t.sessions, 0), 2)                      AS revenue_per_session,
        ROUND(r.revenue / NULLIF(o.total_orders, 0), 2)                  AS avg_order_value,
        t.avg_bounce_pct,
        o.avg_rating
    FROM monthly_traffic t
    JOIN monthly_orders  o ON t.month = o.month
    JOIN monthly_revenue r ON t.month = r.month
    ORDER BY t.month, revenue_per_session DESC
""")

result_q12.show(20)
result_q12.toPandas()

+-------+--------------+-------+--------+---------------+-------------+------------+--------------+--------------------+-------------------+---------------+--------------+----------+
|  month|traffic_source| region|sessions|unique_visitors|unique_buyers|total_orders|       revenue|visitor_to_buyer_pct|revenue_per_session|avg_order_value|avg_bounce_pct|avg_rating|
+-------+--------------+-------+--------+---------------+-------------+------------+--------------+--------------------+-------------------+---------------+--------------+----------+
|2013-01|      referral|Central|   28134|          21470|          277|         282| 9.130867703E7|                1.29|            3245.49|      323789.63|          0.51|      3.98|
|2013-01|      referral|   East|   28134|          21470|          567|         581| 9.130867703E7|                2.64|            3245.49|      157157.79|          0.51|      3.94|
|2013-01|      referral|   West|   28134|          21470|          345|         356| 

,month,traffic_source,region,sessions,unique_visitors,unique_buyers,total_orders,revenue,visitor_to_buyer_pct,revenue_per_session,avg_order_value,avg_bounce_pct,avg_rating
0,2013-01,referral,Central,28134,21470,277,282,91308677.03,1.29,3245.49,323789.63,0.51,3.98
1,2013-01,referral,East,28134,21470,567,581,91308677.03,2.64,3245.49,157157.79,0.51,3.94
2,2013-01,referral,West,28134,21470,345,356,91308677.03,1.61,3245.49,256485.05,0.51,3.89
3,2013-01,social_media,Central,28776,21863,277,282,91308677.03,1.27,3173.08,323789.63,0.43,3.98
4,2013-01,social_media,East,28776,21863,567,581,91308677.03,2.59,3173.08,157157.79,0.43,3.94
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2083,2022-12,social_media,East,127109,96713,142,142,52454936.20,0.15,412.68,369400.96,0.46,3.94
2084,2022-12,social_media,West,127109,96713,74,74,52454936.20,0.08,412.68,708850.49,0.46,4.14
2085,2022-12,organic_search,Central,221159,170057,95,95,52454936.20,0.06,237.18,552157.22,0.44,3.98
2086,2022-12,organic_search,East,221159,170057,142,142,52454936.20,0.08,237.18,369400.96,0.44,3.94


Advanced Promotion ROI: Phân tích hiệu quả từng đợt khuyến mãi với discount stacking. lift_pct dương và cao = khuyến mãi hiệu quả thực sự tăng doanh thu. lift_pct âm = chạy khuyến mãi nhưng doanh thu vẫn thấp hơn ngày thường → cần xem lại thời điểm hoặc kênh chạy promo.

In [27]:

result_q14 = spark.sql("""
    WITH promo_periods AS (
        SELECT
            promo_id,
            promo_name,
            promo_type,
            promo_channel,
            discount_value,
            stackable_flag,
            min_order_value,
            start_date,
            end_date,
            DATEDIFF(end_date, start_date) + 1         AS promo_duration_days
        FROM promotions
    ),
    revenue_in_promo AS (
        SELECT
            p.promo_id,
            p.promo_name,
            p.promo_type,
            p.promo_channel,
            p.discount_value,
            p.stackable_flag,
            p.promo_duration_days,
            ROUND(SUM(s.Revenue), 2)                   AS promo_revenue,
            ROUND(SUM(s.COGS), 2)                      AS promo_cogs,
            COUNT(s.Date)                              AS active_days_with_sales,
            ROUND(AVG(s.Revenue), 2)                   AS avg_daily_revenue
        FROM promo_periods p
        JOIN sales s ON s.Date BETWEEN p.start_date AND p.end_date
        GROUP BY p.promo_id, p.promo_name, p.promo_type,
                 p.promo_channel, p.discount_value,
                 p.stackable_flag, p.promo_duration_days
    ),
    baseline AS (
        SELECT ROUND(AVG(Revenue), 2) AS baseline_daily_revenue FROM sales
    )
    SELECT
        r.promo_id,
        r.promo_name,
        r.promo_type,
        r.promo_channel,
        r.discount_value                               AS discount_pct,
        r.stackable_flag,
        r.promo_duration_days,
        r.promo_revenue,
        ROUND(r.promo_revenue - r.promo_cogs, 2)       AS promo_gross_profit,
        r.avg_daily_revenue,
        b.baseline_daily_revenue,
        ROUND(r.avg_daily_revenue - b.baseline_daily_revenue, 2)        AS daily_revenue_lift,
        ROUND((r.avg_daily_revenue / b.baseline_daily_revenue - 1) * 100, 2) AS lift_pct,
        RANK() OVER (
            PARTITION BY r.promo_type
            ORDER BY (r.avg_daily_revenue / b.baseline_daily_revenue) DESC
        )                                              AS rank_in_promo_type
    FROM revenue_in_promo r
    CROSS JOIN baseline b
    ORDER BY lift_pct DESC
""")

result_q14.show(20, truncate=False)
result_q14.toPandas()

+----------+------------------+----------+-------------+------------+--------------+-------------------+--------------+------------------+-----------------+----------------------+------------------+--------+------------------+
|promo_id  |promo_name        |promo_type|promo_channel|discount_pct|stackable_flag|promo_duration_days|promo_revenue |promo_gross_profit|avg_daily_revenue|baseline_daily_revenue|daily_revenue_lift|lift_pct|rank_in_promo_type|
+----------+------------------+----------+-------------+------------+--------------+-------------------+--------------+------------------+-----------------+----------------------+------------------+--------+------------------+
|PROMO-0021|Spring Sale 2017  |percentage|all_channels |12.0        |0             |31                 |2.2803909351E8|2.25925624E7      |7356099.79       |4286584.03            |3069515.76        |71.61   |1                 |
|PROMO-0011|Spring Sale 2015  |percentage|all_channels |12.0        |1             |31      

,promo_id,promo_name,promo_type,promo_channel,discount_pct,stackable_flag,promo_duration_days,promo_revenue,promo_gross_profit,avg_daily_revenue,baseline_daily_revenue,daily_revenue_lift,lift_pct,rank_in_promo_type
0,PROMO-0021,Spring Sale 2017,percentage,all_channels,12.0,0,31,2.280391e+08,22592562.40,7356099.79,4286584.03,3069515.76,71.61,1
1,PROMO-0011,Spring Sale 2015,percentage,all_channels,12.0,1,31,2.264538e+08,25734142.92,7304962.21,4286584.03,3018378.18,70.41,2
2,PROMO-0017,Spring Sale 2016,percentage,all_channels,12.0,0,31,2.224252e+08,21886187.57,7175006.67,4286584.03,2888422.64,67.38,3
3,PROMO-0007,Spring Sale 2014,percentage,email,12.0,1,31,2.138120e+08,21473680.39,6897161.53,4286584.03,2610577.50,60.90,4
4,PROMO-0028,Mid-Year Sale 2018,percentage,social_media,18.0,0,30,1.935863e+08,10250019.24,6452878.14,4286584.03,2166294.11,50.54,5
5,PROMO-0027,Spring Sale 2018,percentage,all_channels,12.0,0,31,1.990225e+08,23376620.84,6420079.06,4286584.03,2133495.03,49.77,6
6,PROMO-0022,Mid-Year Sale 2017,percentage,online,18.0,0,30,1.917846e+08,9193083.44,6392818.78,4286584.03,2106234.75,49.14,7
7,PROMO-0001,Spring Sale 2013,percentage,email,12.0,1,31,1.877521e+08,21613622.86,6056520.20,4286584.03,1769936.17,41.29,8
8,PROMO-0018,Mid-Year Sale 2016,percentage,online,18.0,0,30,1.799618e+08,9576850.13,5998726.48,4286584.03,1712142.45,39.94,9
9,PROMO-0012,Mid-Year Sale 2015,percentage,social_media,18.0,0,30,1.710466e+08,11764527.63,5701553.47,4286584.03,1414969.44,33.01,10


Q15.inventory + returns + orders

In [28]:
result_q15 = spark.sql("""
WITH return_rate_by_product AS (
    SELECT
        r.product_id,
        COUNT(r.return_id)                                  AS total_returns,
        SUM(r.return_quantity)                              AS returned_qty,
        ROUND(AVG(r.refund_amount), 2)                      AS avg_refund,
        -- lý do trả hàng phổ biến nhất
        FIRST(r.return_reason)                              AS top_return_reason
    FROM returns r
    GROUP BY r.product_id
),
inventory_latest AS (
    SELECT product_id, product_name, category, segment,
           stock_on_hand, sell_through_rate, stockout_flag,
           overstock_flag, reorder_flag, days_of_supply
    FROM inventory
    WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM inventory)
)
SELECT
    i.category, i.segment,
    i.product_name,
    i.stock_on_hand,
    ROUND(i.sell_through_rate * 100, 2)                     AS sell_through_pct,
    i.days_of_supply,
    COALESCE(r.total_returns, 0)                            AS total_returns,
    COALESCE(r.returned_qty, 0)                             AS returned_qty,
    COALESCE(r.avg_refund, 0)                               AS avg_refund,
    r.top_return_reason,
    CASE WHEN i.stockout_flag  = 1 THEN 'YES' ELSE 'NO' END AS is_stockout,
    CASE WHEN i.overstock_flag = 1 THEN 'YES' ELSE 'NO' END AS is_overstock,
    CASE WHEN i.reorder_flag   = 1 THEN 'YES' ELSE 'NO' END AS needs_reorder
FROM inventory_latest i
LEFT JOIN return_rate_by_product r ON i.product_id = r.product_id
ORDER BY total_returns DESC, stock_on_hand ASC
""")

result_q15.show(20, truncate=False)
result_q15.toPandas()

+----------+-----------+-----------------+-------------+----------------+--------------+-------------+------------+----------+-----------------+-----------+------------+-------------+
|category  |segment    |product_name     |stock_on_hand|sell_through_pct|days_of_supply|total_returns|returned_qty|avg_refund|top_return_reason|is_stockout|is_overstock|needs_reorder|
+----------+-----------+-----------------+-------------+----------------+--------------+-------------+------------+----------+-----------------+-----------+------------+-------------+
|Outdoor   |Activewear |HanoiStreet RP-80|2617         |4.84            |590.3         |619          |1732        |1936.65   |wrong_size       |YES        |YES         |NO           |
|Outdoor   |Activewear |HanoiStreet RP-79|2673         |4.74            |602.9         |582          |1577        |1846.36   |defective        |YES        |YES         |NO           |
|Streetwear|Everyday   |SaigonFlex UC-69 |2370         |0.13            |23700.0

,category,segment,product_name,stock_on_hand,sell_through_pct,days_of_supply,total_returns,returned_qty,avg_refund,top_return_reason,is_stockout,is_overstock,needs_reorder
0,Outdoor,Activewear,HanoiStreet RP-80,2617,4.84,590.3,619,1732,1936.65,wrong_size,YES,YES,NO
1,Outdoor,Activewear,HanoiStreet RP-79,2673,4.74,602.9,582,1577,1846.36,defective,YES,YES,NO
2,Streetwear,Everyday,SaigonFlex UC-69,2370,0.13,23700.0,441,1169,13862.76,wrong_size,YES,YES,NO
3,Streetwear,Performance,VietMotion UE-06,1939,0.97,3061.6,402,1139,14148.04,wrong_size,YES,YES,NO
4,Streetwear,Performance,MekongFit UE-18,1913,0.36,8198.6,401,1079,5132.30,defective,YES,YES,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...
419,Streetwear,Performance,UrbanVN UE-02,15,6.25,450.0,0,0,0.00,None,YES,YES,NO
420,Outdoor,Premium,VietMode RS-39,16,30.43,68.6,0,0,0.00,None,YES,NO,NO
421,Streetwear,Balanced,SaigonFlex UM-57,22,50.00,30.0,0,0,0.00,None,YES,NO,NO
422,Streetwear,Everyday,BambooCraft UC-15,27,3.57,810.0,0,0,0.00,None,NO,YES,NO


Q16.Thời gian giao hàng trung bình theo vùng & order_source


In [9]:
result_q16 = spark.sql("""
WITH delivery_time AS (
    SELECT
        o.order_id,
        o.order_source,
        o.device_type,
        o.order_status,
        g.region,
        DATEDIFF(s.delivery_date, s.ship_date)      AS shipping_days,
        DATEDIFF(s.ship_date,     o.order_date)     AS processing_days,
        DATEDIFF(s.delivery_date, o.order_date)     AS total_fulfillment_days,
        s.shipping_fee
    FROM orders o
    JOIN shipments  s ON o.order_id  = s.order_id
    JOIN geography  g ON o.zip       = g.zip
    WHERE o.order_status = 'delivered'
)
SELECT
    region,
    order_source,
    COUNT(*)                                        AS total_orders,
    ROUND(AVG(processing_days), 1)                  AS avg_processing_days,
    ROUND(AVG(shipping_days), 1)                    AS avg_shipping_days,
    ROUND(AVG(total_fulfillment_days), 1)           AS avg_total_days,
    ROUND(AVG(shipping_fee), 2)                     AS avg_shipping_fee,
    SUM(CASE WHEN total_fulfillment_days <= 3
             THEN 1 ELSE 0 END)                     AS fast_delivery_count,
    ROUND(SUM(CASE WHEN total_fulfillment_days <= 3
             THEN 1 ELSE 0 END) * 100.0
             / COUNT(*), 2)                         AS fast_delivery_pct
FROM delivery_time
GROUP BY region, order_source
ORDER BY avg_total_days ASC
""")
result_q16.show()
result_q16.toPandas()


+-------+--------------+------------+-------------------+-----------------+--------------+----------------+-------------------+-----------------+
| region|  order_source|total_orders|avg_processing_days|avg_shipping_days|avg_total_days|avg_shipping_fee|fast_delivery_count|fast_delivery_pct|
+-------+--------------+------------+-------------------+-----------------+--------------+----------------+-------------------+-----------------+
|Central|organic_search|       41279|                1.5|              4.5|           6.0|            4.75|               5169|            12.52|
|   East|email_campaign|       28010|                1.5|              4.5|           6.0|            4.94|               3512|            12.54|
|   West|      referral|       13420|                1.5|              4.5|           6.0|            5.33|               1732|            12.91|
|Central|   paid_search|       32258|                1.5|              4.5|           6.0|            4.74|               40

,region,order_source,total_orders,avg_processing_days,avg_shipping_days,avg_total_days,avg_shipping_fee,fast_delivery_count,fast_delivery_pct
0,Central,organic_search,41279,1.5,4.5,6.0,4.75,5169,12.52
1,East,email_campaign,28010,1.5,4.5,6.0,4.94,3512,12.54
2,West,referral,13420,1.5,4.5,6.0,5.33,1732,12.91
3,Central,paid_search,32258,1.5,4.5,6.0,4.74,4096,12.70
4,East,paid_search,51548,1.5,4.5,6.0,4.82,6417,12.45
5,East,referral,23404,1.5,4.5,6.0,4.77,3001,12.82
6,Central,direct,11587,1.5,4.5,6.0,4.87,1375,11.87
7,West,email_campaign,16066,1.5,4.5,6.0,5.34,2026,12.61
8,Central,social_media,29543,1.5,4.5,6.0,4.79,3640,12.32
9,East,direct,19018,1.5,4.5,6.0,4.90,2355,12.38


Q17. Return Analysis: Tỷ lệ hoàn trả theo lý do, danh mục & phương thức thanh toán

In [29]:
result_q17 = spark.sql("""
WITH return_detail AS (
    SELECT
        r.return_id,
        r.return_reason,
        r.return_quantity,
        r.refund_amount,
        p.category,
        p.segment,
        o.payment_method,
        o.device_type,
        o.order_source
    FROM returns     r
    JOIN order_items oi ON r.order_id    = oi.order_id
                        AND r.product_id = oi.product_id
    JOIN products    p  ON r.product_id  = p.product_id
    JOIN orders      o  ON r.order_id    = o.order_id
),
order_totals AS (
    SELECT category, COUNT(DISTINCT o.order_id) AS total_orders
    FROM orders o
    JOIN order_items oi ON o.order_id   = oi.order_id
    JOIN products    p  ON oi.product_id = p.product_id
    GROUP BY category
)
SELECT
    rd.category,
    rd.return_reason,
    rd.payment_method,
    COUNT(*)                                        AS return_count,
    SUM(rd.return_quantity)                         AS total_returned_qty,
    ROUND(SUM(rd.refund_amount), 2)                 AS total_refund,
    ROUND(AVG(rd.refund_amount), 2)                 AS avg_refund_per_return,
    ROUND(COUNT(*) * 100.0 / ot.total_orders, 2)   AS return_rate_pct
FROM return_detail rd
JOIN order_totals  ot ON rd.category = ot.category
GROUP BY rd.category, rd.return_reason, rd.payment_method, ot.total_orders
ORDER BY return_rate_pct DESC
""")
result_q17.show()
result_q17.toPandas()

+----------+----------------+--------------+------------+------------------+-------------+---------------------+---------------+
|  category|   return_reason|payment_method|return_count|total_returned_qty| total_refund|avg_refund_per_return|return_rate_pct|
+----------+----------------+--------------+------------+------------------+-------------+---------------------+---------------+
|   Outdoor|      wrong_size|   credit_card|        2515|              6840|1.338346669E7|              5321.46|           1.25|
|      GenZ|      wrong_size|   credit_card|         357|              1059|   2129286.88|              5964.39|           0.96|
|    Casual|      wrong_size|   credit_card|         227|               630|   2557893.86|             11268.25|           0.96|
|Streetwear|      wrong_size|   credit_card|        3709|             10149|6.849504013E7|             18467.25|           0.95|
|   Outdoor|       defective|   credit_card|        1493|              4164|   8174654.34|       

,category,return_reason,payment_method,return_count,total_returned_qty,total_refund,avg_refund_per_return,return_rate_pct
0,Outdoor,wrong_size,credit_card,2515,6840,13383466.69,5321.46,1.25
1,GenZ,wrong_size,credit_card,357,1059,2129286.88,5964.39,0.96
2,Casual,wrong_size,credit_card,227,630,2557893.86,11268.25,0.96
3,Streetwear,wrong_size,credit_card,3709,10149,68495040.13,18467.25,0.95
4,Outdoor,defective,credit_card,1493,4164,8174654.34,5475.32,0.74
...,...,...,...,...,...,...,...,...
95,GenZ,late_delivery,bank_transfer,12,26,38101.81,3175.15,0.03
96,Outdoor,late_delivery,bank_transfer,68,219,451469.98,6639.26,0.03
97,Casual,late_delivery,bank_transfer,6,22,98129.82,16354.97,0.03
98,Casual,changed_mind,bank_transfer,6,10,44121.69,7353.61,0.03


Q18.Payment Intelligence: Giá trị đơn hàng & installment theo kênh và thiết bị

In [30]:
result_q18 = spark.sql("""
WITH order_value AS (
    SELECT
        o.order_id,
        o.order_source,
        o.device_type,
        o.payment_method,
        py.payment_value,
        py.installments,
        ROUND(SUM(oi.quantity * oi.unit_price), 2)          AS gross_order_value,
        ROUND(SUM(oi.discount_amount), 2)                   AS total_discount,
        ROUND(SUM(oi.quantity * oi.unit_price)
              - SUM(oi.discount_amount), 2)                 AS net_order_value,
        COUNT(DISTINCT oi.product_id)                       AS items_count
    FROM orders      o
    JOIN order_items oi ON o.order_id  = oi.order_id
    JOIN payments    py ON o.order_id  = py.order_id
    GROUP BY o.order_id, o.order_source, o.device_type,
             o.payment_method, py.payment_value, py.installments
)
SELECT
    order_source,
    device_type,
    payment_method,
    COUNT(*)                                                AS total_orders,
    ROUND(AVG(net_order_value), 2)                          AS avg_order_value,
    ROUND(AVG(items_count), 2)                              AS avg_items_per_order,
    ROUND(AVG(total_discount), 2)                           AS avg_discount,
    ROUND(AVG(installments), 2)                             AS avg_installments,
    SUM(CASE WHEN installments > 1 THEN 1 ELSE 0 END)      AS installment_orders,
    ROUND(SUM(CASE WHEN installments > 1
              THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2)    AS installment_rate_pct,
    RANK() OVER (
        PARTITION BY device_type
        ORDER BY AVG(net_order_value) DESC
    )                                                       AS rank_by_device
FROM order_value
GROUP BY order_source, device_type, payment_method
ORDER BY avg_order_value DESC
""")
result_q18.show()
result_q18.toPandas()

+--------------+-----------+--------------+------------+---------------+-------------------+------------+----------------+------------------+--------------------+--------------+
|  order_source|device_type|payment_method|total_orders|avg_order_value|avg_items_per_order|avg_discount|avg_installments|installment_orders|installment_rate_pct|rank_by_device|
+--------------+-----------+--------------+------------+---------------+-------------------+------------+----------------+------------------+--------------------+--------------+
|email_campaign|     tablet|        paypal|        1769|       25451.48|               1.11|      1315.7|            3.86|              1241|               70.15|             1|
|        direct|     tablet|     apple_pay|         766|       25205.87|                1.1|     1173.27|             3.9|               544|               71.02|             2|
|        direct|     mobile|     apple_pay|        2400|       25180.17|                1.1|     1105.11|     

,order_source,device_type,payment_method,total_orders,avg_order_value,avg_items_per_order,avg_discount,avg_installments,installment_orders,installment_rate_pct,rank_by_device
0,email_campaign,tablet,paypal,1769,25451.48,1.11,1315.70,3.86,1241,70.15,1
1,direct,tablet,apple_pay,766,25205.87,1.10,1173.27,3.90,544,71.02,2
2,direct,mobile,apple_pay,2400,25180.17,1.10,1105.11,3.94,1709,71.21,1
3,direct,mobile,paypal,3546,25026.00,1.10,1131.37,3.90,2479,69.91,2
4,organic_search,tablet,cod,4125,24988.90,1.11,1128.54,1.00,0,0.00,3
...,...,...,...,...,...,...,...,...,...,...,...
85,social_media,tablet,cod,2898,23495.90,1.10,1143.43,1.00,0,0.00,27
86,referral,tablet,bank_transfer,471,23449.49,1.14,1127.77,3.88,315,66.88,28
87,email_campaign,tablet,bank_transfer,578,23215.02,1.12,1211.46,3.77,385,66.61,29
88,email_campaign,tablet,apple_pay,1205,23159.47,1.11,1179.55,3.84,833,69.13,30


In [32]:
import os
import matplotlib.pyplot as plt

output_path = "/Users/tranduykhoa/Downloads/Data/results"

os.makedirs(output_path, exist_ok=True)

def export_query_result(result_df, query_name, show_rows=20):
    pdf = result_df.limit(50).toPandas()

    csv_path = f"{output_path}/{query_name}.csv"
    pdf.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"CSV saved: {csv_path}")

    fig, ax = plt.subplots(figsize=(max(10, len(pdf.columns) * 1.5),
                                    min(20, len(pdf) * 0.4 + 1.5)))
    ax.axis("off")
    tbl = ax.table(
        cellText  = pdf.values,
        colLabels = pdf.columns,
        cellLoc   = "center",
        loc       = "center"
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8)
    tbl.auto_set_column_width(col=list(range(len(pdf.columns))))

    for j in range(len(pdf.columns)):
        tbl[0, j].set_facecolor("#2C3E50")
        tbl[0, j].set_text_props(color="white", fontweight="bold")

    for i in range(1, len(pdf) + 1):
        for j in range(len(pdf.columns)):
            tbl[i, j].set_facecolor("#EAF2FF" if i % 2 == 0 else "white")

    plt.title(query_name.replace("_", " ").upper(),
              fontsize=11, fontweight="bold", pad=12)
    plt.tight_layout()

    img_path = f"{output_path}/{query_name}.png"
    plt.savefig(img_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"PNG saved: {img_path}")

    return pdf

all_queries = [
    (result_q1,  "Q01_doanh_thu_theo_thang"),
    (result_q2,  "Q02_khach_hang_theo_kenh"),
    (result_q3,  "Q03_top_san_pham_bien_loi_nhuan"),
    (result_q4,  "Q04_rating_theo_danh_muc"),
    (result_q5,  "Q05_web_traffic_theo_nguon"),
    (result_q6,  "Q06_khach_hang_theo_vung"),
    (result_q7,  "Q07_hieu_qua_khuyen_mai"),
    (result_q8,  "Q08_window_rank_san_pham"),
    (result_q9,  "Q09_rfm_analysis"),
    (result_q10, "Q10_full_pipeline_join"),
    (result_q11, "Q11_inventory_toan_dien"),
    (result_q12, "Q12_cohort_retention"),
    (result_q14, "Q14_funnel_traffic_to_revenue"),
    (result_q15, "Q15_promotion_roi_lift"),
    (result_q16, "Q16_delivery_performance"),
    (result_q17, "Q17_return_analysis"),
    (result_q18, "Q18_payment_intelligence"),
]




📊 Đang xuất Q01_doanh_thu_theo_thang...
✅ CSV saved: /Users/tranduykhoa/Downloads/Data/results/Q01_doanh_thu_theo_thang.csv
✅ PNG saved: /Users/tranduykhoa/Downloads/Data/results/Q01_doanh_thu_theo_thang.png

📊 Đang xuất Q02_khach_hang_theo_kenh...


26/06/13 20:02:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 20:02:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 20:02:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 20:02:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 20:02:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 20:02:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 2

✅ CSV saved: /Users/tranduykhoa/Downloads/Data/results/Q02_khach_hang_theo_kenh.csv
✅ PNG saved: /Users/tranduykhoa/Downloads/Data/results/Q02_khach_hang_theo_kenh.png

📊 Đang xuất Q03_top_san_pham_bien_loi_nhuan...
✅ CSV saved: /Users/tranduykhoa/Downloads/Data/results/Q03_top_san_pham_bien_loi_nhuan.csv
✅ PNG saved: /Users/tranduykhoa/Downloads/Data/results/Q03_top_san_pham_bien_loi_nhuan.png

📊 Đang xuất Q04_rating_theo_danh_muc...
✅ CSV saved: /Users/tranduykhoa/Downloads/Data/results/Q04_rating_theo_danh_muc.csv
✅ PNG saved: /Users/tranduykhoa/Downloads/Data/results/Q04_rating_theo_danh_muc.png

📊 Đang xuất Q05_web_traffic_theo_nguon...
✅ CSV saved: /Users/tranduykhoa/Downloads/Data/results/Q05_web_traffic_theo_nguon.csv
✅ PNG saved: /Users/tranduykhoa/Downloads/Data/results/Q05_web_traffic_theo_nguon.png

📊 Đang xuất Q06_khach_hang_theo_vung...


26/06/13 20:02:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 20:02:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 20:02:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 20:02:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 20:02:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 20:02:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 2

✅ CSV saved: /Users/tranduykhoa/Downloads/Data/results/Q06_khach_hang_theo_vung.csv
✅ PNG saved: /Users/tranduykhoa/Downloads/Data/results/Q06_khach_hang_theo_vung.png

📊 Đang xuất Q07_hieu_qua_khuyen_mai...


...

26/06/13 20:03:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 20:03:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 20:03:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 20:03:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 20:03:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


✅ PNG saved: /Users/tranduykhoa/Downloads/Data/results/Q11_inventory_toan_dien.png

📊 Đang xuất Q12_cohort_retention...
✅ CSV saved: /Users/tranduykhoa/Downloads/Data/results/Q12_cohort_retention.csv
✅ PNG saved: /Users/tranduykhoa/Downloads/Data/results/Q12_cohort_retention.png

📊 Đang xuất Q14_funnel_traffic_to_revenue...
✅ CSV saved: /Users/tranduykhoa/Downloads/Data/results/Q14_funnel_traffic_to_revenue.csv
✅ PNG saved: /Users/tranduykhoa/Downloads/Data/results/Q14_funnel_traffic_to_revenue.png

📊 Đang xuất Q15_promotion_roi_lift...
✅ CSV saved: /Users/tranduykhoa/Downloads/Data/results/Q15_promotion_roi_lift.csv
✅ PNG saved: /Users/tranduykhoa/Downloads/Data/results/Q15_promotion_roi_lift.png

📊 Đang xuất Q16_delivery_performance...


✅ CSV saved: /Users/tranduykhoa/Downloads/Data/results/Q16_delivery_performance.csv
✅ PNG saved: /Users/tranduykhoa/Downloads/Data/results/Q16_delivery_performance.png

📊 Đang xuất Q17_return_analysis...


✅ CSV saved: /Users/tranduykhoa/Downloads/Data/results/Q17_return_analysis.csv
✅ PNG saved: /Users/tranduykhoa/Downloads/Data/results/Q17_return_analysis.png

📊 Đang xuất Q18_payment_intelligence...


✅ CSV saved: /Users/tranduykhoa/Downloads/Data/results/Q18_payment_intelligence.csv
✅ PNG saved: /Users/tranduykhoa/Downloads/Data/results/Q18_payment_intelligence.png

🎉 Hoàn tất! Kiểm tra thư mục: /Users/tranduykhoa/Downloads/Data/results
